# AIMLCZG546 — Software Engineering for Machine Learning
# Assignment II — Implementation, Code Quality & Quality Assurance

## Group No: 049

| Sl. No | BITS ID | Name | Contribution (Qualitative) | % Contribution |
| --- | --- | --- | --- | --- |
| 1 | `<FILL BITS ID>` | `<FILL NAME>` | Production package design — data ingestion, feature engineering, configuration and the research-vs-production refactor (Tasks 1, 2) | 25 |
| 2 | `<FILL BITS ID>` | `<FILL NAME>` | Logging, error handling, code formatting and linting setup — black / flake8 / isort reports (Tasks 3, 4) | 25 |
| 3 | `<FILL BITS ID>` | `<FILL NAME>` | REST API design and implementation with FastAPI, request/response schemas and status codes (Task 5) | 25 |
| 4 | `<FILL BITS ID>` | `<FILL NAME>` | Quality assurance — pytest suite, ML-specific tests, model & data quality metrics, production testing and security note (Tasks 6–9) | 25 |

---

## The system in one paragraph

**E-commerce Session Conversion Prediction.** A shopper is browsing the store right now.
Given what they have done so far in this session — pages viewed, time spent, items in the
cart, whether they are a returning customer, the device they are on — the system predicts
the probability that the session ends in a purchase. High-intent sessions can be offered
live chat or a nudge; low-intent ones are left alone. This continues the e-commerce domain
from Assignment I, rebuilt as a small, self-contained, production-shaped codebase.

**Model:** logistic regression (scaled) → a calibrated probability, not just a yes/no.
Calibration matters here because the business acts on *how likely*, not just *likely*.

---

## How to run everything

```bash
pip install -r requirements.txt
python -m ecom.data_ingestion   # 1. build the demo dataset
python -m ecom.train            # 2. train + evaluate + save the artifact
python -m ecom.monitor          # 3. data quality + drift report
python -m pytest                # 4. run the full QA suite
uvicorn ecom.api:app --reload   # 5. serve the model at http://127.0.0.1:8000/docs
```

---

## Repository layout

```
assignment-2/
├── 049.ipynb              <- this notebook (the submission)
├── ecom/                  <- PRODUCTION package
│   ├── config.py            paths, data contract, thresholds
│   ├── logging_setup.py     one logging config for everything
│   ├── exceptions.py        typed errors
│   ├── data_ingestion.py    dataset generation + CSV loading      (OOP)
│   ├── data_validation.py   schema / missing / drift checks       (OOP)
│   ├── features.py          feature engineering                   (functional)
│   ├── train.py             training pipeline + artifact          (OOP)
│   ├── metrics.py           model quality scorecard               (functional)
│   ├── predict.py           inference wrapper                     (OOP)
│   ├── monitor.py           scheduled data quality check
│   ├── schemas.py           API request/response contracts
│   └── api.py               FastAPI service
├── research/              <- RESEARCH code, kept as the "before" exhibit
│   ├── research_prototype.ipynb
│   └── prototype.py
├── tests/                 <- pytest suite (61 tests)
├── reports/               <- lint reports, test report, data quality report
├── artifacts/             <- model.joblib, metrics.json
├── data/                  <- generated CSVs
└── JOURNEY.md             <- how this project was built, step by step
```


In [1]:
# Setup: run everything from the project root.
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
print("Python  :", sys.version.split()[0])
print("Working :", Path.cwd().name)

Python  : 3.12.10
Working : assignment-2


---
# Objective 1 — Implementation and Code Sharing  [5 Marks]

## Task 1 — Modular design using OOP and functional principles

The application is split so each module has **one job** and can be tested on its own.
Two styles are used deliberately:

| Layer | Module | Style | Why this style |
| --- | --- | --- | --- |
| Ingestion | `data_ingestion.py` | **OOP** (`DataIngestor`, `SessionDataGenerator`) | Holds state (a path, a random seed); a class keeps it tidy and swappable |
| Validation | `data_validation.py` | **OOP** (`DataValidator` → `ValidationReport`) | Configurable with a schema; returns a rich object |
| Features | `features.py` | **Functional** (pure functions) | Same input → same output, no hidden state, so training and serving cannot drift apart |
| Metrics | `metrics.py` | **Functional** | Stateless maths — a class would add nothing |
| Training | `train.py` | **OOP** (`ModelTrainer`) | Owns the fitted pipeline and how it is saved |
| Inference | `predict.py` | **OOP** (`ConversionPredictor`) | Loads the artifact once, then serves many requests |
| API | `api.py` + `schemas.py` | Thin layer | Translates HTTP ↔ the predictor; contains no ML logic |

The important rule: **the API knows nothing about feature engineering, and feature
engineering knows nothing about HTTP.** Each can change without touching the other.

In [2]:
# The module map, straight from the code's own docstrings.
for path in sorted(Path("ecom").glob("*.py")):
    if path.name == "__init__.py":
        continue
    first_line = path.read_text(encoding="utf-8").splitlines()[0].strip('"')
    print(f"{path.name:20s} {first_line}")

api.py               REST API exposing the conversion model.
config.py            Central place for paths, schema and constants. No logic here.
data_ingestion.py    Data ingestion: create the demo dataset and load it from disk.
data_validation.py   Data quality gate.
exceptions.py        Custom errors so callers can react to *why* something failed.
features.py          Feature engineering - written as pure functions.
logging_setup.py     One logging config for the whole package.
metrics.py           Model quality metrics.
monitor.py           Data quality monitor.
predict.py           Inference: load the saved artifact and score sessions.
schemas.py           Request/response contracts for the REST API.
train.py             Model training: raw CSV -> validated data -> features -> saved artifact.


In [3]:
# The same four layers, called one after the other - each is independently usable.
from ecom.data_ingestion import DataIngestor
from ecom.data_validation import DataValidator
from ecom.features import build_features
from ecom.predict import ConversionPredictor

raw = DataIngestor().load()                 # 1. ingestion
report = DataValidator().validate(raw)      # 2. validation
features = build_features(raw)              # 3. feature engineering
predictor = ConversionPredictor().load()    # 4. inference

print("\nraw shape      :", raw.shape)
print("feature shape  :", features.shape)
print("model version  :", predictor.version)

2026-08-07 13:35:29,445 | INFO     | ecom.data_ingestion | Loading data from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\data\sessions_train.csv


2026-08-07 13:35:29,454 | INFO     | ecom.data_ingestion | Loaded 5000 rows, 9 columns


2026-08-07 13:35:29,457 | INFO     | ecom.data_validation | Data validation passed (schema conformance=1.00)


2026-08-07 13:35:29,458 | WARNING  | ecom.data_validation | Data quality warning: 'avg_item_price' has 1.0% missing values


2026-08-07 13:35:29,460 | WARNING  | ecom.features | Imputing 50 missing avg_item_price values with 0


2026-08-07 13:35:29,466 | INFO     | ecom.features | Built 12 features for 5000 rows


2026-08-07 13:35:30,949 | INFO     | ecom.predict | Model v1.0.0 loaded from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\artifacts\model.joblib



raw shape      : (5000, 9)
feature shape  : (5000, 12)
model version  : 1.0.0


In [4]:
raw.head()

,session_id,pages_viewed,session_duration_min,cart_items,avg_item_price,is_returning_user,discount_applied,device,converted
0,s000000,9,12.20,2,723.48,0,0,desktop,1
1,s000001,7,9.17,0,57.89,0,0,desktop,0
2,s000002,9,10.73,1,468.53,0,1,mobile,1
3,s000003,7,6.27,0,181.62,0,0,mobile,0
4,s000004,7,4.27,1,1549.23,1,0,desktop,1


In [5]:
features.head()

,pages_viewed,session_duration_min,cart_items,avg_item_price,is_returning_user,discount_applied,pages_per_minute,cart_value,has_cart,device_desktop,device_mobile,device_tablet
0,9,12.20,2,723.48,0,0,0.681818,1446.96,1,1,0,0
1,7,9.17,0,57.89,0,0,0.688299,0.00,0,1,0,0
2,9,10.73,1,468.53,0,1,0.767263,468.53,1,0,1,0
3,7,6.27,0,181.62,0,0,0.962861,0.00,0,0,1,0
4,7,4.27,1,1549.23,1,0,1.328273,1549.23,1,1,0,0


## Task 2 — Research code vs Production code

**The component:** the *conversion model training + scoring* path.

| | Research (`research/research_prototype.ipynb`) | Production (`ecom/` package) |
| --- | --- | --- |
| Goal | Answer "does this work?" once | Run unattended, every day, for years |
| File path | Hardcoded string | `ecom/config.py` |
| Reproducibility | No seed — every run differs | Fixed `RANDOM_SEED` |
| Feature logic | Typed inline in a cell | `features.py`, shared by training *and* serving |
| Feature order | Hand-copied list | Saved inside the artifact, re-applied at inference |
| Scaling | Applied in one cell, forgotten in another | Inside the sklearn `Pipeline` — impossible to forget |
| Bad input | Cryptic error deep in sklearn | `DataValidationError` with the offending column |
| Output | `print()` | `logging` with levels, to console **and** file |
| Correctness | "It looked right" | 61 automated tests |
| Reuse | Copy-paste the cell | `pip install` and import |

Open `research/research_prototype.ipynb` to see the "before" version in full.
The cell below shows the single most important difference in one number: **reproducibility.**

In [6]:
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from ecom.config import RANDOM_SEED, TARGET, TEST_SIZE
from ecom.train import ModelTrainer

X, y = build_features(raw), raw[TARGET]


def research_style():
    """Prototype: no seed, no scaling - a different answer every run."""
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
    with warnings.catch_warnings():
        # Unscaled input makes the solver struggle - the prototype never noticed.
        warnings.simplefilter("ignore")
        model = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
    return round(accuracy_score(y_te, model.predict(X_te)), 4)


def production_style():
    """Package: fixed seed, scaling inside the pipeline - same answer every run."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    return round(ModelTrainer().fit(X_tr, y_tr).score(X_te, y_te), 4)


print("research  (3 runs):", [research_style() for _ in range(3)], "<- drifts")
print("production (3 runs):", [production_style() for _ in range(3)], "<- identical")

2026-08-07 13:35:30,982 | WARNING  | ecom.features | Imputing 50 missing avg_item_price values with 0


2026-08-07 13:35:30,989 | INFO     | ecom.features | Built 12 features for 5000 rows


research  (3 runs): [0.722, 0.719, 0.731] <- drifts
2026-08-07 13:35:31,412 | INFO     | ecom.train | Training on 4000 rows, 12 features


2026-08-07 13:35:31,427 | INFO     | ecom.train | Training on 4000 rows, 12 features


2026-08-07 13:35:31,440 | INFO     | ecom.train | Training on 4000 rows, 12 features


production (3 runs): [0.716, 0.716, 0.716] <- identical


## Task 3 — Error handling and logging

`ecom/logging_setup.py` configures logging **once** for the whole package: every module
calls `get_logger(__name__)`, output goes to the console *and* to `logs/app.log`, and the
message carries a timestamp, level and module name.

Log levels are used to mean specific things:

| Level | Used for | Example in this codebase |
| --- | --- | --- |
| `INFO` | Normal progress worth recording | `"Loaded 5000 rows, 9 columns"` |
| `WARNING` | Survivable, but someone should know | `"Imputing 50 missing avg_item_price values with 0"`, `"Drift detected in 'pages_viewed' (PSI=1.690)"` |
| `ERROR` | The operation failed | `"Data file not found: ..."`, `"Model artifact not found at ..."` |

Errors are typed (`ecom/exceptions.py`) so a caller can react to *why* something failed
rather than parsing a message. **Five critical modules** carry full handling:

1. `data_ingestion.DataIngestor.load` → `DataIngestionError` (missing / corrupt / empty file)
2. `data_validation.DataValidator.validate` → `DataValidationError` (broken contract)
3. `predict.ConversionPredictor.load` → `ModelNotReadyError` (missing / unloadable artifact)
4. `train.ModelTrainer.save` → `RuntimeError` (saving before fitting)
5. `api.py` → maps those errors to honest HTTP status codes (503, not 500)

In [7]:
# Each failure below is caught deliberately, with a typed error and a log line.
from ecom.data_validation import DataValidator
from ecom.exceptions import DataIngestionError, DataValidationError, ModelNotReadyError


def missing_file():
    DataIngestor(Path("data/does_not_exist.csv")).load()


def broken_schema():
    DataValidator().validate(raw.drop(columns=["cart_items"]))


def negative_values():
    bad = raw.copy()
    bad.loc[0, "pages_viewed"] = -5
    DataValidator().validate(bad)


def no_model():
    ConversionPredictor(Path("artifacts/not_here.joblib")).load()


def save_before_fit():
    ModelTrainer().save()


for name, scenario in [
    ("missing file", missing_file),
    ("broken schema", broken_schema),
    ("negative values", negative_values),
    ("no model artifact", no_model),
    ("save before fit", save_before_fit),
]:
    try:
        scenario()
    except (DataIngestionError, DataValidationError, ModelNotReadyError, RuntimeError) as exc:
        print(f"  -> handled [{name}] {type(exc).__name__}: {exc}\n")

2026-08-07 13:35:31,457 | INFO     | ecom.data_ingestion | Loading data from data\does_not_exist.csv


2026-08-07 13:35:31,458 | ERROR    | ecom.data_ingestion | Data file not found: data\does_not_exist.csv


  -> handled [missing file] DataIngestionError: Data file not found: data\does_not_exist.csv

2026-08-07 13:35:31,461 | ERROR    | ecom.data_validation | Data validation FAILED: missing column 'cart_items'


  -> handled [broken schema] DataValidationError: missing column 'cart_items'

2026-08-07 13:35:31,463 | ERROR    | ecom.data_validation | Data validation FAILED: 'pages_viewed' contains negative values


  -> handled [negative values] DataValidationError: 'pages_viewed' contains negative values

2026-08-07 13:35:31,464 | ERROR    | ecom.predict | Model artifact not found at artifacts\not_here.joblib


  -> handled [no model artifact] ModelNotReadyError: No model at artifacts\not_here.joblib. Train first.

  -> handled [save before fit] RuntimeError: Nothing to save - call fit() first



In [8]:
# The log file proves the levels are actually being written.
log_lines = Path("logs/app.log").read_text(encoding="utf-8").splitlines()
for level in ("INFO", "WARNING", "ERROR"):
    matches = [line for line in log_lines if f"| {level}" in line]
    print(f"{level:8s} {len(matches):4d} lines   e.g. {matches[-1][:110] if matches else '-'}")

INFO      400 lines   e.g. 2026-08-07 13:35:31,457 | INFO     | ecom.data_ingestion | Loading data from data\does_not_exist.csv
WARNING   104 lines   e.g. 2026-08-07 13:35:30,982 | WARNING  | ecom.features | Imputing 50 missing avg_item_price values with 0
ERROR      56 lines   e.g. 2026-08-07 13:35:31,464 | ERROR    | ecom.predict | Model artifact not found at artifacts\not_here.joblib


## Task 4 — Code formatting and linting

Three tools, configured in `pyproject.toml` (black, isort) and `setup.cfg` (flake8):

| Tool | Job |
| --- | --- |
| **black** | Formats code to one non-negotiable style (88 columns) |
| **isort** | Sorts imports into standard-library / third-party / local blocks |
| **flake8** | Catches unused imports, undefined names, style violations |

**Before:** the research prototype (`research/prototype.py`) plus the production code as
first written — 52 flake8 violations, 2 files needing reformatting, 2 files with unsorted
imports.

**After:** the production package (`ecom/` and `tests/`) — **zero violations**, all files
already formatted, all imports sorted. `research/` is deliberately left un-linted; it is
the exhibit.

In [9]:
print(Path("reports/lint_before.txt").read_text(encoding="utf-8"))

 LINT REPORT - BEFORE
 Command: flake8 / black --check / isort --check research ecom tests

--- flake8 ----------------------------------------------------
ecom\predict.py:63:89: E501 line too long (91 > 88 characters)
research\prototype.py:5:1: F401 'os' imported but unused
research\prototype.py:5:1: F401 'sys' imported but unused
research\prototype.py:5:1: F401 'json' imported but unused
research\prototype.py:5:10: E401 multiple imports on one line
research\prototype.py:7:1: F401 'numpy as np' imported but unused
research\prototype.py:7:13: E272 multiple spaces before keyword
research\prototype.py:10:1: F403 'from sklearn.metrics import *' used; unable to detect undefined names
research\prototype.py:12:18: E201 whitespace after '('
research\prototype.py:12:44: E202 whitespace before ')'
research\prototype.py:17:23: E225 missing whitespace around operator
research\prototype.py:19:15: E225 missing whitespace around operator
research\prototype.py:19:33: E225 missing whitespace around op

In [10]:
print(Path("reports/lint_after.txt").read_text(encoding="utf-8"))

 LINT REPORT - AFTER
 Production code only (research/ is kept as the 'before' exhibit)
 Command: flake8 / black --check / isort --check ecom tests

--- flake8 ----------------------------------------------------
(exit code: 0 -> 0 means zero violations)

--- black --check ---------------------------------------------
All done!
20 files would be left unchanged.

--- isort --check-only ----------------------------------------
(no output = all imports correctly sorted)



In [11]:
# Re-run the tools live so the "after" state is verifiable, not just a saved file.
for tool in (["flake8", "ecom", "tests"],
             ["black", "--check", "ecom", "tests"],
             ["isort", "--check-only", "ecom", "tests"]):
    result = subprocess.run([sys.executable, "-m", *tool], capture_output=True, text=True)
    status = "PASS" if result.returncode == 0 else "FAIL"
    print(f"[{status}] python -m {' '.join(tool)}")

[PASS] python -m flake8 ecom tests


[PASS] python -m black --check ecom tests


[PASS] python -m isort --check-only ecom tests


## Task 5 — REST API with FastAPI

`ecom/api.py` exposes the model over HTTP. Design decisions:

| Endpoint | Method | Purpose | Status codes |
| --- | --- | --- | --- |
| `/health` | GET | Liveness probe | `200` always — reports `degraded` rather than dying, so a missing model does not get the container killed |
| `/model-info` | GET | Which model is serving | `200`, `503` if no model |
| `/metrics` | GET | Last training scorecard | `200`, `404` if never trained |
| `/predict` | POST | Score one session | `200`, `422` invalid input, `503` no model |
| `/predict/batch` | POST | Score up to 100 sessions | `200`, `422`, `503` |

Good-API-design choices made here:

- **Nouns and clear verbs** — `POST /predict`, not `GET /doPrediction`.
- **Typed request/response schemas** (`ecom/schemas.py`) with pydantic, so the OpenAPI
  docs at `/docs` are generated from the code and can never go stale.
- **Validation at the edge** — ranges (`pages_viewed` 0–500), an enum for `device`, and
  `extra="forbid"` so unknown fields are rejected instead of silently ignored.
- **Bounded payloads** — batch capped at 100 sessions.
- **Honest status codes** — a missing model is `503 Service Unavailable` (try again later),
  not `500 Internal Server Error` (we are broken).
- **Model loaded once at startup** via the lifespan hook, not per request.

In [12]:
from fastapi.testclient import TestClient

from ecom.api import app

client = TestClient(app)
client.__enter__()          # runs startup: loads the model

def show(method, url, payload=None):
    response = getattr(client, method)(url, json=payload) if payload else getattr(client, method)(url)
    print(f"{method.upper():5s} {url:22s} -> {response.status_code}")
    print(json.dumps(response.json(), indent=2)[:420])
    print("-" * 70)

show("get", "/health")
show("get", "/model-info")

2026-08-07 13:35:33,398 | INFO     | ecom.predict | Model v1.0.0 loaded from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\artifacts\model.joblib


GET   /health                -> 200
{
  "status": "ok",
  "model_loaded": true,
  "api_version": "1.0.0"
}
----------------------------------------------------------------------
GET   /model-info            -> 200
{
  "model_version": "1.0.0",
  "features": [
    "pages_viewed",
    "session_duration_min",
    "cart_items",
    "avg_item_price",
    "is_returning_user",
    "discount_applied",
    "pages_per_minute",
    "cart_value",
    "has_cart",
    "device_desktop",
    "device_mobile",
    "device_tablet"
  ],
  "n_features": 12
}
----------------------------------------------------------------------


C:\Users\ravir\AppData\Local\Programs\Python\Python312-arm64\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [13]:
session = {
    "session_id": "s000123",
    "pages_viewed": 12,
    "session_duration_min": 8.5,
    "cart_items": 2,
    "avg_item_price": 799.0,
    "is_returning_user": 1,
    "discount_applied": 0,
    "device": "mobile",
}
show("post", "/predict", session)
show("post", "/predict/batch", {"sessions": [session, {**session, "cart_items": 0}]})

2026-08-07 13:35:33,443 | INFO     | ecom.features | Built 12 features for 1 rows


2026-08-07 13:35:33,448 | INFO     | ecom.predict | Scored 1 session(s)


2026-08-07 13:35:33,449 | INFO     | ecom.api | session=s000123 prob=0.9025 band=high


POST  /predict               -> 200
{
  "session_id": "s000123",
  "conversion_probability": 0.9025,
  "will_convert": true,
  "intent_band": "high",
  "model_version": "1.0.0"
}
----------------------------------------------------------------------
2026-08-07 13:35:33,455 | INFO     | ecom.features | Built 12 features for 2 rows


2026-08-07 13:35:33,457 | INFO     | ecom.predict | Scored 2 session(s)


2026-08-07 13:35:33,458 | INFO     | ecom.api | Batch scored 2 sessions


POST  /predict/batch         -> 200
{
  "count": 2,
  "predictions": [
    {
      "session_id": "s000123",
      "conversion_probability": 0.9025,
      "will_convert": true,
      "intent_band": "high",
      "model_version": "1.0.0"
    },
    {
      "session_id": "s000123",
      "conversion_probability": 0.4184,
      "will_convert": false,
      "intent_band": "medium",
      "model_version": "1.0.0"
    }
  ]
}
----------------------------------------------------------------------


In [14]:
# Input validation in action - every one of these is rejected with 422.
bad_payloads = {
    "negative pages": {**session, "pages_viewed": -1},
    "wrong type": {**session, "pages_viewed": "twelve"},
    "unknown device": {**session, "device": "smart-fridge"},
    "missing field": {k: v for k, v in session.items() if k != "device"},
    "injected field": {**session, "sql": "DROP TABLE users"},
}
for label, payload in bad_payloads.items():
    print(f"{label:18s} -> {client.post('/predict', json=payload).status_code}")

print("oversized batch    ->",
      client.post("/predict/batch", json={"sessions": [session] * 101}).status_code)
print("unknown route      ->", client.get("/does-not-exist").status_code)

negative pages     -> 422
wrong type         -> 422
unknown device     -> 422
missing field      -> 422
injected field     -> 422
oversized batch    -> 422
unknown route      -> 404


In [15]:
# The OpenAPI contract is generated from the code itself.
spec = client.get("/openapi.json").json()
for route, methods in spec["paths"].items():
    for method, detail in methods.items():
        print(f"{method.upper():5s} {route:22s} {detail['summary']}")

GET   /health                Health
GET   /model-info            Model Info
GET   /metrics               Metrics
POST  /predict               Predict
POST  /predict/batch         Predict Batch


---
# Objective 2 — Quality Assurance  [5 Marks]

## Task 6 — Test types

**Three** types of test, 61 tests in total, all with `pytest`:

| # | Type | File | What it protects |
| --- | --- | --- | --- |
| 1 | **Data validation tests** | `tests/test_data_validation.py` | The input contract — missing columns, wrong dtypes, too many nulls, negative values, distribution drift |
| 2 | **Unit tests** | `tests/test_features.py`, `tests/test_data_ingestion.py` | Individual functions — one behaviour per test, plus the ingestion error paths |
| 3 | **Integration tests** | `tests/test_api.py` | The whole stack together — HTTP → schema → features → model → response |

Plus the ML-specific tests of Task 7 (`test_training.py`, `test_inference.py`).

In [16]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--no-header", "--color=no", "-p", "no:cacheprovider"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])

PASSED                      [  1%]
tests/test_api.py::test_model_info PASSED                                [  3%]
tests/test_api.py::test_metrics_endpoint PASSED                          [  4%]
tests/test_api.py::test_predict_returns_200_and_a_valid_body PASSED      [  6%]
tests/test_api.py::test_batch_predict PASSED                             [  8%]
tests/test_api.py::test_invalid_input_returns_422[pages_viewed--1] PASSED [  9%]
tests/test_api.py::test_invalid_input_returns_422[pages_viewed-twelve] PASSED [ 11%]
tests/test_api.py::test_invalid_input_returns_422[cart_items-10000] PASSED [ 13%]
tests/test_api.py::test_invalid_input_returns_422[device-smart-fridge] PASSED [ 14%]
tests/test_api.py::test_invalid_input_returns_422[avg_item_price--50] PASSED [ 16%]
tests/test_api.py::test_invalid_input_returns_422[session_id-] PASSED    [ 18%]
tests/test_api.py::test_missing_required_field_returns_422 PASSED        [ 19%]
tests/test_api.py::test_unknown_field_is_rejected PASSED            

In [17]:
# Test counts per file.
counts = {}
for line in result.stdout.splitlines():
    if "::" in line and ("PASSED" in line or "FAILED" in line):
        counts[line.split("::")[0]] = counts.get(line.split("::")[0], 0) + 1
for file, n in sorted(counts.items()):
    print(f"{n:4d}  {file}")
print(f"{sum(counts.values()):4d}  TOTAL")

  16  tests/test_api.py
   6  tests/test_data_ingestion.py
  10  tests/test_data_validation.py
   8  tests/test_features.py
  15  tests/test_inference.py
   6  tests/test_training.py
  61  TOTAL


## Task 7 — ML-specific tests

Ordinary software tests ask "did the function return the right value?". ML tests have to
ask harder questions, because there is no single right answer.

### (a) Testing model **training** — `tests/test_training.py`

| Test | The question it answers |
| --- | --- |
| `test_loss_decreases_with_more_training` | Is the optimiser actually *learning*, or just running? |
| `test_model_can_overfit_a_small_batch` | Can the pipeline memorise 20 rows? If not, something is wired wrong |
| `test_trained_model_beats_the_release_gate` | Is it good enough to ship (accuracy ≥ 0.70, F1 ≥ 0.60)? |
| `test_model_beats_a_majority_class_baseline` | Is it better than always guessing the common class? |
| `test_training_is_reproducible` | Same seed → same model? |

### (b) Testing model **inference** — `tests/test_inference.py`

| Test | Kind | The question it answers |
| --- | --- | --- |
| `test_output_shape_matches_input` | Shape | N sessions in → N predictions out |
| `test_probability_is_in_range` | Range | Is every probability inside [0, 1]? |
| `test_more_cart_items_increases_...` | **Directional** | More items in the cart *must* raise conversion probability |
| `test_returning_user_is_more_likely...` | **Directional** | A returning shopper *must* score higher |
| `test_discount_increases_...` | **Directional** | A discount *must* raise the probability |
| `test_session_id_does_not_change_...` | **Invariance** | Changing an irrelevant field must change nothing |
| `test_device_casing_does_not_change_...` | **Invariance** | `"MOBILE"` and `"mobile"` are the same device |
| `test_row_order_does_not_change_...` | **Invariance** | Batch order must not affect results |

The next cells demonstrate the two most important ones live.

In [18]:
# (a) Training: the loss must go down as the optimiser gets more budget.
steps = (1, 2, 5, 10, 50, 200)
losses = ModelTrainer().loss_curve(X, y, steps=steps)

print("\n  iterations   train log-loss")
for step, loss in zip(steps, losses):
    bar = "#" * int(loss * 60)
    print(f"  {step:9d}   {loss:.5f}  {bar}")
print(f"\n  decreasing? {all(b <= a + 1e-6 for a, b in zip(losses, losses[1:]))}")

2026-08-07 13:35:37,387 | INFO     | ecom.train | Loss curve [1, 2, 5, 10, 50, 200] -> [0.585299, 0.565118, 0.555881, 0.554273, 0.554211, 0.554211]



  iterations   train log-loss
          1   0.58530  ###################################
          2   0.56512  #################################
          5   0.55588  #################################
         10   0.55427  #################################
         50   0.55421  #################################
        200   0.55421  #################################

  decreasing? True


In [19]:
# (a) Training: the pipeline must be able to memorise a tiny balanced batch.
index = list(y[y == 0].index[:10]) + list(y[y == 1].index[:10])
batch_X, batch_y = X.loc[index], y.loc[index]

overfit = ModelTrainer(max_iter=5000)._new_pipeline()
overfit.named_steps["model"].set_params(C=1e6)      # regularisation off
overfit.fit(batch_X, batch_y)

print(f"\naccuracy on the 20-row batch: {overfit.score(batch_X, batch_y):.3f}  (expect 1.000)")


accuracy on the 20-row batch: 1.000  (expect 1.000)


In [20]:
# (b) Inference: directional and invariance checks.
def session_with(**overrides):
    base = dict(session_id="t", pages_viewed=8, session_duration_min=5.0, cart_items=1,
                avg_item_price=500.0, is_returning_user=0, discount_applied=0, device="mobile")
    base.update(overrides)
    return base

def probability(**overrides):
    return predictor.predict([session_with(**overrides)])[0]["conversion_probability"]

print("\nDIRECTIONAL (probability must go up)")
print(f"  cart_items      0 -> 3 : {probability(cart_items=0):.4f} -> {probability(cart_items=3):.4f}")
print(f"  returning user  0 -> 1 : {probability(is_returning_user=0):.4f} -> {probability(is_returning_user=1):.4f}")
print(f"  discount        0 -> 1 : {probability(discount_applied=0):.4f} -> {probability(discount_applied=1):.4f}")

print("\nINVARIANCE (probability must not change)")
print(f"  session_id  x vs y     : {probability(session_id='x'):.4f} == {probability(session_id='y'):.4f}")
print(f"  device casing          : {probability(device='mobile'):.4f} == {probability(device='MOBILE'):.4f}")


DIRECTIONAL (probability must go up)
2026-08-07 13:35:37,411 | INFO     | ecom.features | Built 12 features for 1 rows


2026-08-07 13:35:37,414 | INFO     | ecom.predict | Scored 1 session(s)


2026-08-07 13:35:37,417 | INFO     | ecom.features | Built 12 features for 1 rows


2026-08-07 13:35:37,420 | INFO     | ecom.predict | Scored 1 session(s)


  cart_items      0 -> 3 : 0.1515 -> 0.9067
2026-08-07 13:35:37,425 | INFO     | ecom.features | Built 12 features for 1 rows


2026-08-07 13:35:37,427 | INFO     | ecom.predict | Scored 1 session(s)


... (11 outputs left out)

2026-08-07 13:35:37,464 | INFO     | ecom.predict | Scored 1 session(s)


2026-08-07 13:35:37,468 | INFO     | ecom.features | Built 12 features for 1 rows


2026-08-07 13:35:37,470 | INFO     | ecom.predict | Scored 1 session(s)


  device casing          : 0.3781 == 0.3781


## Task 8 — Metrics

### (a) Model quality — 4 metrics, because one number always lies

| Metric | Plain English | Why it is here |
| --- | --- | --- |
| **Accuracy** | How often the yes/no call is right | The headline number — but it flatters a lazy model on imbalanced data |
| **F1** | Balance of precision and recall on *converters* | The class we actually care about is the smaller one |
| **ROC-AUC** | Can it *rank* a converter above a non-converter? | Threshold-free; survives a change of cut-off |
| **Brier score** | **Calibration** — is a "70%" really 70%? Lower is better | The business acts on the probability, not the label |

Plus a **calibration table**: predicted rate vs actual rate, bucket by bucket.

### (b) Data quality — 3 metrics

| Metric | Plain English | Where |
| --- | --- | --- |
| **Schema conformance** | Fraction of contract columns present with the right type (1.0 = perfect) | `DataValidator._check_schema` |
| **Missing value rate** | Null fraction per column; above 5% the run is rejected | `DataValidator._check_missing` |
| **PSI drift score** | Has live traffic moved away from training data? >0.20 = significant drift | `population_stability_index` |

In [21]:
# (a) Model quality - retrain end to end and read the scorecard.
from ecom.train import run_training

metrics = run_training()
print("\n=== MODEL QUALITY ===")
for name, value in metrics["model_quality"].items():
    print(f"  {name:14s} {value}")
print("\n  release gate:", metrics["release_gate"])

2026-08-07 13:35:37,476 | INFO     | ecom.data_ingestion | Loading data from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\data\sessions_train.csv


2026-08-07 13:35:37,482 | INFO     | ecom.data_ingestion | Loaded 5000 rows, 9 columns


2026-08-07 13:35:37,484 | INFO     | ecom.data_validation | Data validation passed (schema conformance=1.00)


2026-08-07 13:35:37,484 | WARNING  | ecom.data_validation | Data quality warning: 'avg_item_price' has 1.0% missing values


2026-08-07 13:35:37,486 | WARNING  | ecom.features | Imputing 50 missing avg_item_price values with 0


2026-08-07 13:35:37,491 | INFO     | ecom.features | Built 12 features for 5000 rows


... (3 outputs left out)

2026-08-07 13:35:37,520 | INFO     | ecom.train | Model saved to C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\artifacts\model.joblib


2026-08-07 13:35:37,580 | INFO     | ecom.train | Loss curve [1, 2, 5, 10, 50, 200] -> [0.583821, 0.56358, 0.554139, 0.552615, 0.552584, 0.552584]


2026-08-07 13:35:37,582 | INFO     | ecom.train | Metrics written to C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\artifacts\metrics.json



=== MODEL QUALITY ===
  accuracy       0.716
  f1             0.683
  precision      0.7034
  recall         0.6638
  roc_auc        0.7813
  brier_score    0.1899

  release gate: {'min_accuracy': 0.7, 'min_f1': 0.6, 'passed': True}


In [22]:
# (a) Calibration: does a predicted 70% really convert 70% of the time?
pd.DataFrame(metrics["calibration"])

,bucket,count,predicted_rate,actual_rate
0,0.0-0.2,197,0.1309,0.1574
1,0.2-0.4,263,0.2964,0.2662
2,0.4-0.6,222,0.4993,0.5450
3,0.6-0.8,200,0.6971,0.7150
4,0.8-1.0,118,0.8814,0.8136


In [23]:
# (b) Data quality: validation + drift, on today's live traffic.
from ecom.monitor import run_data_quality_check

quality = run_data_quality_check()

print("\n=== DATA QUALITY ===")
print("  schema conformance :", quality["validation"]["schema_conformance"])
print("  validation passed  :", quality["validation"]["passed"])
print("  warnings           :", quality["validation"]["warnings"])
print("\n  missing value rate per column:")
for column, rate in quality["validation"]["missing_rates"].items():
    print(f"    {column:22s} {rate:.2%}")

2026-08-07 13:35:37,596 | INFO     | ecom.data_ingestion | Loading data from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\data\sessions_train.csv


2026-08-07 13:35:37,601 | INFO     | ecom.data_ingestion | Loaded 5000 rows, 9 columns


2026-08-07 13:35:37,602 | INFO     | ecom.data_ingestion | Loading data from C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\data\sessions_live.csv


2026-08-07 13:35:37,605 | INFO     | ecom.data_ingestion | Loaded 2500 rows, 9 columns


2026-08-07 13:35:37,608 | INFO     | ecom.data_validation | Data validation passed (schema conformance=1.00)


2026-08-07 13:35:37,608 | WARNING  | ecom.data_validation | Data quality warning: 'avg_item_price' has 1.0% missing values


... (3 outputs left out)

2026-08-07 13:35:37,614 | INFO     | ecom.data_validation | No drift in 'avg_item_price' (PSI=0.001)


2026-08-07 13:35:37,615 | INFO     | ecom.monitor | Data quality report written to C:\dev\ai-ml\software-engineering-for-machine-learning\assignments\assignment-2\reports\data_quality.json


2026-08-07 13:35:37,615 | WARNING  | ecom.monitor | Retraining suggested - drifted: ['pages_viewed', 'session_duration_min']



=== DATA QUALITY ===
  schema conformance : 1.0
  validation passed  : True
  warnings           : ["'avg_item_price' has 1.0% missing values"]

  missing value rate per column:
    session_id             0.00%
    pages_viewed           0.00%
    session_duration_min   0.00%
    cart_items             0.00%
    avg_item_price         1.00%
    is_returning_user      0.00%
    discount_applied       0.00%
    device                 0.00%


In [24]:
print("=== DRIFT (PSI: >0.20 = significant) ===\n")
for column, psi in quality["drift_psi"].items():
    flag = "DRIFT" if psi > 0.20 else "ok"
    print(f"  {column:22s} {psi:8.4f}  {flag}")
print("\n  columns needing attention:", quality["drifted_columns"])
print("  action: the live feed has shorter, more mobile-heavy sessions than the")
print("          training data -> schedule a retrain before accuracy degrades.")

=== DRIFT (PSI: >0.20 = significant) ===

  pages_viewed             1.6902  DRIFT
  session_duration_min     0.8272  DRIFT
  cart_items               0.0012  ok
  avg_item_price           0.0014  ok

  columns needing attention: ['pages_viewed', 'session_duration_min']
  action: the live feed has shorter, more mobile-heavy sessions than the
          training data -> schedule a retrain before accuracy degrades.


## Task 9 — Testing in production, and a security consideration

### (a) Testing / experimentation in production — **shadow deployment, then canary**

Offline metrics are measured on data we already have. Production has traffic we have never
seen. So the new model earns its way in, in three stages:

**Stage 1 — Shadow deployment (no risk).**
Deploy the new model *alongside* the current one. Every live request is sent to both.
Only the current model's answer is returned to the customer; the new model's answer is
logged and thrown away. After a week we compare the two on real traffic: do the
predictions agree? is the new one better calibrated? is it fast enough? Nobody is
affected if it is wrong — which is exactly the point.

*Concretely here:* the API would call both predictors and log
`{session_id, prob_current, prob_shadow, latency_ms}`, then join those rows against the
actual purchase outcome the next day.

**Stage 2 — Canary release (small, reversible risk).**
Route **5%** of sessions to the new model, keeping 95% on the old one. Watch a small,
pre-agreed set of guardrails: error rate, p99 latency, and the real conversion rate of the
5% group. If any guardrail trips, route back to 100% old — a config change, not a redeploy.
If it holds for 24 hours, step to 25%, then 50%, then 100%.

**Stage 3 — A/B test (measuring business value, not model accuracy).**
Once the new model is safe, the remaining question is whether it *makes money*. Split
users 50/50, run for two weeks, and compare the business metric — revenue per session —
with a significance test. A model with better AUC that does not move revenue is not worth
the operational cost of keeping.

**Why this order:** shadow answers "is it broken?", canary answers "is it safe?",
A/B answers "is it better?". Each stage is cheap to reverse and each one gates the next.

### (b) Security consideration — **input validation at the API boundary**

The most exposed surface of this system is `POST /predict`: it is public, it accepts
JSON from a browser, and it feeds that JSON straight into a numerical model.

**The risk.** Unvalidated fields let a caller send `pages_viewed = 10_000_000`, a string
where a number belongs, or an unexpected `device` value. In the mild case this produces a
nonsense probability that some downstream system trusts. In the worse case it is a probe:
an attacker sweeps extreme values to map the decision boundary, learns that a high
`cart_items` reliably raises the score, and games the ranking — a model-inversion /
adversarial-input attack. Oversized batch payloads are also a cheap denial-of-service.

**The mitigation, already implemented in `ecom/schemas.py`:**

| Control | Implementation | Attack it blunts |
| --- | --- | --- |
| Type + range limits | `pages_viewed: int = Field(ge=0, le=500)`, and similar on every numeric field | Out-of-distribution probing, absurd inputs |
| Closed value set | `device: Literal["mobile", "desktop", "tablet"]` | Injection through a free-text field |
| Unknown fields rejected | `model_config = {"extra": "forbid"}` | Parameter smuggling into downstream code |
| Bounded payload | `sessions: list[...] = Field(max_length=100)` | Denial of service via huge requests |
| Length limits | `session_id: str = Field(max_length=64)` | Log flooding, buffer-style abuse |

Everything that fails these checks is rejected with **422** *before the model is ever
called* — proven by the 422 tests in `tests/test_api.py`.

**Beyond input validation** (noted, not implemented in this assignment): an API key or
JWT on the inference endpoints; per-client rate limiting to slow down boundary probing;
read-only file permissions on `artifacts/model.joblib` so a compromised process cannot
swap the model; and returning only the intent band rather than the raw probability to
external callers, since precise probabilities leak the most information about the model.

---
## Summary — requirement checklist

| # | Requirement | Where it is done | Evidence |
| --- | --- | --- | --- |
| 1 | OOP / functional modular design | `ecom/` — 11 modules | Task 1 module map |
| 2 | Research code vs production code | `research/` vs `ecom/` | Task 2 reproducibility demo |
| 3 | Error handling + logging in ≥3 modules | 5 modules, INFO/WARNING/ERROR | Task 3 log counts |
| 4 | black / flake8 / isort with before–after | `reports/lint_before.txt`, `lint_after.txt` | Task 4: 52 violations → 0 |
| 5 | REST API with FastAPI | `ecom/api.py`, `ecom/schemas.py` | Task 5: 5 endpoints, live calls |
| 6 | ≥2 types of test with pytest | 3 types: data validation, unit, integration | Task 6: 61 tests passing |
| 7a | Model training tests | `tests/test_training.py` | Loss curve + 20-row overfit |
| 7b | Model inference tests | `tests/test_inference.py` | Shape, range, directional, invariance |
| 8a | ≥2 model quality metrics | accuracy, F1, ROC-AUC, Brier + calibration | Task 8a scorecard |
| 8b | ≥2 data quality metrics | schema conformance, missing rate, PSI drift | Task 8b report |
| 9 | Production testing + security | Shadow → canary → A/B; input validation | Task 9 |

The full build story — every decision and why it was made — is in **`JOURNEY.md`**.